# Lab 06. K-means 군집과 PCA

K-means는 군집 내부 제곱거리 합을 최소화한다.

$$J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert^2$$

거리 기반 모델이므로 표준화 전후 결과를 반드시 비교한다. PCA는 고차원 데이터를 시각화하지만 정보 손실이 있다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 루트에서 Notebook을 실행하세요.")

sys.path.insert(0, str(ROOT))
STUDENT_ID = "20260001"  # 반드시 본인 학번으로 변경
print("저장소:", ROOT)
print("실습 학번:", STUDENT_ID)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from src.education.personalized_data import make_student_dataset, student_seed

df = make_student_dataset(STUDENT_ID).drop_duplicates()
columns = ["유동인구", "경쟁점포수", "주차장수", "월임대료", "월매출"]
X = df[columns].copy().fillna(df[columns].median())
display(X.describe().T[["mean", "std", "min", "max"]].round(1))

## 1. 표준화

StandardScaler는 학습 데이터의 평균을 빼고 표준편차로 나눈다. 변환 후 각 변수 평균은 약 0, 표준편차는 약 1이다.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
scaled_summary = pd.DataFrame(X_scaled, columns=columns).agg(["mean", "std"]).T
display(scaled_summary.round(3))

## 2. k 후보 비교

inertia는 k가 증가하면 항상 감소하므로 단독으로 최적 k를 정할 수 없다.
silhouette는 -1에서 1 사이이며 같은 군집에는 가깝고 다른 군집에는 멀수록 높다.

In [ ]:
records = []
seed = student_seed(STUDENT_ID)
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=seed, n_init=20)
    labels = model.fit_predict(X_scaled)
    records.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
    })
scores = pd.DataFrame(records)
display(scores.round(3))

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(scores["k"], scores["inertia"], marker="o", label="inertia")
ax1.set_xlabel("k"); ax1.set_ylabel("inertia")
ax2 = ax1.twinx()
ax2.plot(scores["k"], scores["silhouette"], marker="s", color="orange")
ax2.set_ylabel("silhouette")
plt.title("군집 수 후보 비교")
plt.show()

## 3. 군집 프로필

아래 BEST_K는 예시다. 본인 점수표와 분석 목적을 근거로 바꾼다. 군집 번호에는 순서 의미가 없다.

In [ ]:
BEST_K = int(scores.loc[scores["silhouette"].idxmax(), "k"])
kmeans = KMeans(n_clusters=BEST_K, random_state=seed, n_init=20)
df["군집"] = kmeans.fit_predict(X_scaled)
profile = df.groupby("군집")[columns].agg(["count", "mean"])
print("선택 k:", BEST_K)
display(profile.round(1))

## 4. PCA 2차원 시각화와 적재량

설명분산비율은 각 주성분이 전체 표준화 분산의 얼마를 보존하는지 나타낸다.
적재량은 원래 변수가 주성분 축에 얼마나 기여하는지 보여준다.

In [ ]:
pca = PCA(n_components=2)
points = pca.fit_transform(X_scaled)
loadings = pd.DataFrame(pca.components_.T, index=columns, columns=["PC1", "PC2"])
print("설명분산비율:", pca.explained_variance_ratio_)
display(loadings.round(3))

plt.figure(figsize=(7, 5))
plt.scatter(points[:, 0], points[:, 1], c=df["군집"], cmap="tab10", alpha=.7)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA 투영과 K-means 군집")
plt.show()

## 5. 표준화하지 않은 결과와 비교

원자료 K-means는 값의 규모가 큰 유동인구가 거리를 지배할 가능성이 높다.

In [ ]:
raw_model = KMeans(n_clusters=BEST_K, random_state=seed, n_init=20)
raw_labels = raw_model.fit_predict(X)
agreement = pd.crosstab(df["군집"], raw_labels, rownames=["표준화 군집"], colnames=["원자료 군집"])
display(agreement)

## 6. 독립 연습

1. k를 선택한 근거를 inertia, silhouette와 해석 가능성으로 작성한다.
2. 군집별 평균을 전체 평균 대비 z점수로 표현한다.
3. 각 군집에 “고유동·고비용”처럼 데이터 기반 이름을 붙인다.
4. 변수 하나를 제외하고 군집이 얼마나 바뀌는지 비교한다.
5. PCA 두 축의 누적 설명분산이 충분한지 판단하고 2차원 그림의 한계를 설명한다.

In [ ]:
# TODO: 전체 평균 대비 군집 프로필 z점수
cluster_z_profile = None
display(cluster_z_profile)

# TODO: 변수 제거 민감도 표
sensitivity = None
display(sensitivity)

## 7. 자가점검

- [ ] 표준화 전후 결과를 비교했다.
- [ ] k를 한 지표만으로 선택하지 않았다.
- [ ] 군집 번호를 서열로 해석하지 않았다.
- [ ] 군집 이름의 근거 변수를 제시했다.
- [ ] PCA 설명분산비율과 적재량을 확인했다.